In [ ]:
import pandas as pd
import numpy as np 
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
from pathlib import Path
import glob
import matplotlib.pyplot as plt
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))


## Load reliable cohorts and representative cohorts

In [ ]:
from config.constants import PROJECT_ROOT
cohorts_dir = PROJECT_ROOT / "saved_data" / "cohorts" / "DTB" / "new"

cohorts_init = pd.read_csv(cohorts_dir / "initial_cohorts_summary.csv")
cohorts_first_adms = pd.read_csv(cohorts_dir / "first_adms_cohorts_summary.csv")
reliable_cohorts_init = pd.read_csv(cohorts_dir / "reliable_cohorts_init.csv", header=None, names=["cohort_name"])
reliable_cohorts_first_adms = pd.read_csv(cohorts_dir / "reliable_cohorts_first_adms.csv", header=None, names=["cohort_name"])
cohorts_first_adms_reliable = cohorts_first_adms[cohorts_first_adms["cohort_name"].isin(reliable_cohorts_first_adms["cohort_name"])]


rep_cohorts = pd.read_csv(cohorts_dir / "representative_cohorts_df.csv").cohort_name


In [ ]:
selected_cohorts = set(rep_cohorts) #& set(reliable_cohorts_first_adms.cohort_name)

## Add pr-gain & pr-skill

In [ ]:

results_dir = PROJECT_ROOT / "saved_data" / "results"
def add_pr_metrics(df):
    out = df.copy()
    out["pr_lift"] = out["pr_avg"] / out["target_rate"]
    out["pr_skill"] = (out["pr_avg"] - out["target_rate"]) / (1 - out["target_rate"])
    return out

## Load results

In [ ]:
# Loading results
def load_results(cohorts, models, prefixes, fold=None):

    folds = [fold] if fold is not None else range(5)
    rows = []
    for cohort in cohorts:
        for model in models:
            prefix = next((p for p in prefixes if (Path(f"../saved_data/results/{cohort}/{model}") / p).exists()),None)
            if prefix is None:
                print("no prefix")
                continue
            for f in folds:
                file = (Path(f"../saved_data/results/{cohort}/{model}/{prefix}")
                        / f"fold_{f}" / "agg_int_24" / "impute_fill" / "variant_VMD" / "results_final.csv")
                if not file.exists():
                    print(f"{cohort} - fold_{f} doesn't exist")
                    continue
                output = pd.read_csv(file)
                output["cohort_name"] = cohort
                output["model"] = model
                output["prefix"] = prefix
                output["fold"] = f
                rows.append(output)
                
    results_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    results_df = results_df[["cohort_name","model","prefix","fold","auroc", "auprc", "f1","mcc","recall","precision"]]
    
    for col in ["auroc", "auprc", "f1","mcc","recall","precision"]:
        results_df[col] = pd.to_numeric(results_df[col], errors="coerce")

    for col, new_col in [("auroc", "roc_avg"), ("auprc", "pr_avg"), ("f1", "f1_avg"),("mcc", "mcc_avg"),("recall","recall_avg"),("precision","precision_avg")]:
        results_df[new_col] = results_df.groupby(["cohort_name", "model"])[col].transform("mean") # average of folds
        
    results_df = results_df[["cohort_name", "model", "roc_avg", "pr_avg", "f1_avg","mcc_avg","recall_avg","precision_avg"]].drop_duplicates(["cohort_name", "model"])
    return results_df



In [ ]:
models = ["random_forest"] #"catboost", "random_forest", "xgboost"]


In [ ]:
VARIANT_CONFIG = {
    "Initial_14d":     (["200526", "190526"],   cohorts_init),
    "Corr-top100_14d": (["corr_feat_training"], cohorts_init),
    "MRMR_14d":        (["mrmr_feat_training"], cohorts_init),
    "first_adms_14d":  (["first_adms"],         cohorts_first_adms),
    "first_adms_90d":  (["first_adms_90d"],     cohorts_first_adms),
}

results = {}
for name, (prefixes, cohort_chars) in VARIANT_CONFIG.items():
    print("Loading,", name)
    df = load_results(selected_cohorts, models, prefixes, fold=0)
    df = df.merge(cohort_chars, on="cohort_name")
    df = add_pr_metrics(df)
    results[name] = df

results_initial       = results["Initial_14d"]
results_corr_100      = results["Corr-top100_14d"]
results_mrmr           = results["MRMR_14d"]
results_first_adms     = results["first_adms_14d"]
results_first_adms_90d = results["first_adms_90d"]


## Compare performance across different runs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

metrics = [
    ("roc_avg", "AUROC", "Random Forest AUROC"),
    #("pr_avg", "AUPRC", "Random Forest AUPRC"),
    ("pr_skill", "PR_norm", "Random Forest PR_norm"),
    ("f1_avg", "F1", "Random Forest F1"),
]
labels = ["Initial_14d", "Corr-top100_14d", "MRMR_14d", "first_adms_14d", "first_adms_90d"]
colors = ["#2a78d6", "#008300", "#e87ba4", "#833b1d", "#ed4b06"]
dfs = [results_initial, results_corr_100, results_mrmr, results_first_adms, results_first_adms_90d]

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 7), sharey=False)

for ax, (col, ylabel, title) in zip(axes, metrics):
    variants = [df[col].dropna() for df in dfs]

    bp = ax.boxplot(variants, tick_labels=labels, widths=0.5, patch_artist=True,
                     showfliers=False, medianprops=dict(color="black", linewidth=2))

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.2)
        patch.set_edgecolor(color)
        patch.set_linewidth(2)

    for i, (df, vals, color) in enumerate(zip(dfs, variants, colors), start=1):
        rel = df.loc[vals.index, "reliable"].values if "reliable" in df.columns else np.ones(len(vals), dtype=bool)

        # dodge: reliable points to the left of center, unreliable to the right
        jitter_rel = i - 0.15 + np.random.normal(0, 0.045, size=rel.sum())
        jitter_unrel = i + 0.15 + np.random.normal(0, 0.045, size=(~rel).sum())

        ax.scatter(jitter_rel, vals.values[rel], color=color, alpha=0.8, s=32, zorder=3, marker="o")
        ax.scatter(jitter_unrel, vals.values[~rel], color=color, alpha=0.5, s=45, zorder=3,
                   marker="x", linewidths=1.8)

    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_title(title, fontsize=16)
    ax.tick_params(axis="x", rotation=30, labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='gray', markersize=9, label='Reliable'),
    Line2D([0], [0], marker='x', color='gray', markersize=9, linewidth=0, markeredgewidth=1.8, label='Unreliable'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.08), fontsize=18)

fig.suptitle("Feature selection & admission variants", y=1.16, fontsize=20)

plt.tight_layout()
plt.show()


## Check if the performance differs across reliable cohorts only

comparing reliable vs unreliable
Mann-Whitney U test — the non-parametric equivalent of a t-test, comparing two independent groups' medians without assuming a normal distribution 

(appropriate here since AUROC/PR-skill values across cohorts are not normally distributed and group sizes differ

In [ ]:
from scipy.stats import mannwhitneyu

def compare_reliable_vs_excluded(df, col, alternative="greater", alpha=0.05):
    reliable = df.loc[df.reliable == True, col].dropna()
    excluded = df.loc[df.reliable == False, col].dropna()

    stat, p = mannwhitneyu(reliable, excluded, alternative=alternative)

    n1, n2 = len(reliable), len(excluded)
    effect = stat / (n1 * n2)   # 0.5 = no difference, 1.0 = every reliable > every excluded

    print(f"{col}: reliable median={reliable.median():.3f} (n={n1})  "
          f"excluded median={excluded.median():.3f} (n={n2})")
    print(f"  P(random reliable > random excluded) = {effect:.3f}")
    print(f"  one-sided p (reliable > excluded) = {p:.4g}")


    if p < alpha:
        diff = abs(effect - 0.5)
        size = "large" if diff >= 0.2 else "moderate" if diff >= 0.1 else "small"
        print(f"  -> CONCLUSION: reliable cohorts have significantly higher {col} "
              f"than excluded ones ({size} effect).")
    else:
        print(f"  -> CONCLUSION: no significant evidence that reliable cohorts have "
              f"higher {col} than excluded ones (p={p:.3g} >= {alpha}).")
    print()

    return effect, p

for col in ["roc_avg", "pr_skill"]:
    compare_reliable_vs_excluded(results_first_adms_90d, col)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

def compare_reliable_vs_excluded(df, col, alternative="greater", alpha=0.05, verbose=True):
    reliable = df.loc[df.reliable == True, col].dropna()
    excluded = df.loc[df.reliable == False, col].dropna()
    if len(reliable) < 2 or len(excluded) < 2:
        return None

    stat, p = mannwhitneyu(reliable, excluded, alternative=alternative)
    n1, n2 = len(reliable), len(excluded)
    effect = stat / (n1 * n2)

    if verbose:
        print(f"{col}: reliable median={reliable.median():.3f} (n={n1})  "
              f"excluded median={excluded.median():.3f} (n={n2})")
        print(f"  P(random reliable > random excluded) = {effect:.3f}")
        print(f"  one-sided p (reliable > excluded) = {p:.4g}")
        if p < alpha:
            diff = abs(effect - 0.5)
            size = "large" if diff >= 0.2 else "moderate" if diff >= 0.1 else "small"
            print(f"  -> CONCLUSION: reliable cohorts have significantly higher {col} "
                  f"than excluded ones ({size} effect).")
        else:
            print(f"  -> CONCLUSION: no significant evidence that reliable cohorts have "
                  f"higher {col} than excluded ones (p={p:.3g} >= {alpha}).")
        print()

    return dict(effect=effect, p=p, n_reliable=n1, n_excluded=n2,
                median_reliable=reliable.median(), median_excluded=excluded.median())


variant_dfs = {
    "Initial_14d": results_initial,
    "Corr-top100_14d": results_corr_100,
    "MRMR_14d": results_mrmr,
    "first_adms_14d": results_first_adms,
    "first_adms_90d": results_first_adms_90d,
}

metrics = ["roc_avg", "pr_skill"]

rows = []
for name, df in variant_dfs.items():
    print(f"=== {name} ===")
    for col in metrics:
        res = compare_reliable_vs_excluded(df, col)
        if res:
            rows.append({"variant": name, "metric": col, **res})

summary = pd.DataFrame(rows)

# --- figure: effect size per variant, per metric, with significance stars ---
fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4), sharey=True)
if len(metrics) == 1:
    axes = [axes]

for ax, col in zip(axes, metrics):
    sub = summary[summary.metric == col].reset_index(drop=True)
    x = np.arange(len(sub))
    bar_colors = ["#2a78d6" if p < 0.05 else "#bbbbbb" for p in sub["p"]]

    ax.bar(x, sub["effect"], color=bar_colors, alpha=0.85, edgecolor="black", linewidth=0.8)
    ax.axhline(0.5, color="black", linestyle="--", linewidth=1)

    for xi, (eff, p) in enumerate(zip(sub["effect"], sub["p"])):
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        ax.text(xi, eff + 0.02, stars, ha="center", fontsize=10)

    ax.set_xticks(x)
    ax.set_xticklabels(sub["variant"], rotation=30, ha="right")
    ax.set_ylabel("P(reliable > excluded)")
    ax.set_title(col)
    ax.set_ylim(0.3, 1.0)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

fig.suptitle("Reliable vs Excluded cohorts, per variant", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import fisher_exact

table = pd.crosstab(results_first_adms_90d["reliable"], results_first_adms_90d["learned"])
print(table)
odds_ratio, p = fisher_exact(table, alternative="greater")
print(f"reliable cohorts more likely to show real signal: OR={odds_ratio:.2f}, p={p:.4g}")


## Check which models have genuinely learned sth

Flag cohorts whose AUROC is statistically distinguishable from chance (0.5),

using Hanley-McNeil SE + FDR correction across cohorts (avoids trusting noisy

small-N point estimates as if they were real signal).


In [ ]:
import numpy as np
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

def add_significance(df, auc_col="roc_avg", alpha=0.05, correct=True):
    df = df.copy()
    n_pos, n_neg, auc = df["n_pos_test_estimate"], df["n_neg_test_estimate"], df[auc_col]

    Q1 = auc / (2 - auc)
    Q2 = 2 * auc**2 / (1 + auc)
    var = (auc*(1-auc) + (n_pos-1)*(Q1-auc**2) + (n_neg-1)*(Q2-auc**2)) / (n_pos*n_neg)
    se = np.sqrt(var)

    z = (auc - 0.5) / se
    p = pd.Series(1 - norm.cdf(z), index=df.index) 

    df["auroc_se"] = se
    df["auroc_p"] = p
    if correct:
        valid = p.notna()
        df.loc[valid, "auroc_p_adj"] = multipletests(p[valid], method="fdr_bh")[1]
        df["learned"] = df["auroc_p_adj"] < alpha
    else:
        df["learned"] = df["auroc_p"] < alpha
    return df


results_initial = add_significance(results_initial)
results_corr_100 = add_significance(results_corr_100)
results_mrmr = add_significance(results_mrmr)
results_first_adms = add_significance(results_first_adms)
results_first_adms_90d = add_significance(results_first_adms_90d)




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

def plot_forest_panel(ax, df, auc_col="roc_avg", title="", show_labels=True):
    d = df.dropna(subset=[auc_col, "auroc_se"]).sort_values(auc_col).reset_index(drop=True)
    y = np.arange(len(d))
    ci = 1.96 * d["auroc_se"]

    groups = [
        (True,  True,  "#1a43d9", "#1a43d9", "o"),   # learned, reliable
        (True,  False, "#1a43d9", "#7b7874", "o"),   # learned, unreliable
        (False, True,  "#7b7874", "#1a43d9", "x"),   # not learned, reliable
        (False, False, "#7b7874", "#7b7874", "x"),   # not learned, unreliable
    ]

    for learned_flag, reliable_flag, mcolor, lcolor, marker in groups:
        mask = ((d["learned"] == learned_flag) & (d["reliable"] == reliable_flag)).values
        if mask.sum() == 0:
            continue
        ax.errorbar(d.loc[mask, auc_col], y[mask], xerr=ci[mask], fmt=marker,
                    ecolor=lcolor, mfc=mcolor, mec=mcolor, ms=10, elinewidth=4,
                    capsize=4, capthick=4, mew=2.5)

    ax.axvline(0.5, color="#c7c2bc", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(d["cohort_name"] if show_labels else [], fontsize=24)
    ax.tick_params(axis="x", labelsize=24)
    ax.set_xlabel("AUROC (95% CI)", fontsize=28)
    ax.set_title(title, fontsize=28)


variant_dfs = {
    "Initial_14d": results_initial,
    "Corr-top100_14d": results_corr_100,
    "MRMR_14d": results_mrmr,
    "first_adms_14d": results_first_adms,
    "first_adms_90d": results_first_adms_90d,
}

max_n = max(len(df.dropna(subset=["roc_avg", "auroc_se"])) for df in variant_dfs.values())
fig_height = max(8, max_n * 0.5)

fig, axes = plt.subplots(1, len(variant_dfs), figsize=(6 * len(variant_dfs), fig_height))

for i, (ax, (name, df)) in enumerate(zip(axes, variant_dfs.items())):
    plot_forest_panel(ax, df, auc_col="roc_avg", title=name, show_labels=(i == 0))

legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor="#1a43d9", markeredgecolor="#1a43d9", markersize=14, label='Learned'),
    Line2D([0], [0], marker='x', color="#7b7874", markersize=14, markeredgewidth=2.5, label='Not learned'),
    Line2D([0], [0], color="#1a43d9", linewidth=4, label='Reliable (CI line)'),
    Line2D([0], [0], color="#7b7874", linewidth=4, label='Unreliable (CI line)'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.03), fontsize=28)

plt.tight_layout()
plt.show()


# Select cohorts based on CI of ROC

In [ ]:
# TODO: will come back to this later if we want to technically select cohorts

In [ ]:
chars = pd.read_csv(cohorts_dir / "first_adms_cohorts_summary.csv") 

ASSUMED_AUC = 0.60          # conservative; SE is largest at low AUROC
TARGET_CI_HALFWIDTH = 0.2  # decide this from what you need to detect

npos, nneg = chars['n_pos_test_estimate'], chars['n_neg_test_estimate']
Q1 = ASSUMED_AUC/(2-ASSUMED_AUC); Q2 = 2*ASSUMED_AUC**2/(1+ASSUMED_AUC)
var = (ASSUMED_AUC*(1-ASSUMED_AUC) + (npos-1)*(Q1-ASSUMED_AUC**2)
       + (nneg-1)*(Q2-ASSUMED_AUC**2))/(npos*nneg)
chars['expected_se'] = np.sqrt(var)
chars['expected_ci_halfwidth'] = 1.96 * chars['expected_se']
chars['worth_training'] = chars['expected_ci_halfwidth'] <= TARGET_CI_HALFWIDTH

print(chars['worth_training'].value_counts())


In [ ]:
print(pd.crosstab(chars["reliable"], chars["worth_training"], margins=True))
